# NHANES Graph Misspecification

One edit to the assumed ADMG: a direct edge $Z \to Y$, dropping the assumption
that health insurance affects diabetes only through physical activity.

The edit leaves the query's district hull $\mathbf{R}_\theta^*$ unchanged, so
every pruning decision survives it. It does change the mechanism of $Y$, which
lies inside the hull, so the potencies are not protected. Both are checked
here: the potencies are recomputed from scratch under each graph, and the
pruning decisions are re-derived from each graph.


## Setup

In [1]:
from __future__ import annotations

import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
SUPP_ROOT = ROOT if (ROOT / 'coupled_potency.py').exists() else ROOT.parent
sys.path.insert(0, str(SUPP_ROOT))

DATA_DIR = SUPP_ROOT / 'data'
PV_PATH = DATA_DIR / 'P_V.csv'

from algorithm_evaluation.causal_graph import CausalGraph
from algorithm_evaluation.get_useless import get_useless_experiments_separate

# Five-variable ADMG: balance problems (A), falls (B), insurance (Z),
# physical activity (X), not-diabetes (Y).
VARIABLES = ['A', 'B', 'Z', 'X', 'Y']
EDGES = [('A', 'B'), ('A', 'X'), ('B', 'X'), ('Z', 'X'), ('X', 'Y')]
CONFOUNDERS = [('A', 'B'), ('Z', 'X'), ('X', 'Y')]

# theta = ATE of X on Y. Both counterfactual worlds carry the (W, Z)
# signature ({Y}, {X}), which is all the pruning rules look at.
QUERY_WORLDS = [(frozenset({'Y'}), frozenset({'X'}))]

# (W, Z) signatures of the Table 1 candidates.
CANDIDATE_WORLDS = [
    ('a1_tilde', 'P(X | do(B))',   [(frozenset({'X'}), frozenset({'B'}))]),
    ('a2_tilde', 'P(B | do(A))',   [(frozenset({'B'}), frozenset({'A'}))]),
    ('a3_tilde', 'P(X,Y | do(z))', [(frozenset({'X', 'Y'}), frozenset({'Z'}))]),
    ('a4_tilde', "P(y_x', y'_x)",  [(frozenset({'Y'}), frozenset({'X'})),
                                    (frozenset({'Y'}), frozenset({'X'}))]),
    ('a5_tilde', "P(Y | do(x'))",  [(frozenset({'Y'}), frozenset({'X'}))]),
    ('a6_tilde', 'P(Y | do(x))',   [(frozenset({'Y'}), frozenset({'X'}))]),
]

def pruning_table(cg):
    """Pruning mechanism per candidate: 'Int.', 'ID' or '--'."""
    per_exp, _ = get_useless_experiments_separate(
        cg, QUERY_WORLDS, [w for _, _, w in CANDIDATE_WORLDS])
    return {label: ('Int.' if pe['blocked'] else ('ID' if pe['id'] else '--'))
            for (label, _, _), pe in zip(CANDIDATE_WORLDS, per_exp)}

print('setup OK')

setup OK


## Part 1: Load and discretize

Identical to Part 1 of `../nhanes_pipeline.ipynb`; rewrites the same
`data/P_V.csv`.

In [2]:
SPECIAL_MISSING = {7, 9, 77, 99, 777, 999}

def load_xpt(fname):
    df = pd.read_sas(str(DATA_DIR / fname))
    df.columns = [c.decode() if isinstance(c, bytes) else str(c) for c in df.columns]
    return df

demo = load_xpt('DEMO_J.xpt')
paq  = load_xpt('PAQ_J.xpt')
hiq  = load_xpt('HIQ_J.xpt')
diq  = load_xpt('DIQ_J.xpt')

merged = demo.merge(paq, on='SEQN', how='left') \
             .merge(hiq, on='SEQN', how='left') \
             .merge(diq, on='SEQN', how='left')
merged = merged[pd.to_numeric(merged['RIDAGEYR'], errors='coerce') >= 18].copy()

def clean(s):
    x = pd.to_numeric(s, errors='coerce')
    return x.mask(x.isin(SPECIAL_MISSING))

df = pd.DataFrame(index=merged.index)

hiq011 = clean(merged['HIQ011'])                       # Z: insurance
df['Z'] = np.nan
df.loc[hiq011 == 1, 'Z'] = 1
df.loc[hiq011 == 2, 'Z'] = 0

paq650 = clean(merged['PAQ650'])                       # X: physical activity
paq665 = clean(merged['PAQ665'])
df['X'] = np.nan
df.loc[(paq650 == 1) | (paq665 == 1), 'X'] = 1
df.loc[(paq650 == 2) & (paq665 == 2), 'X'] = 0

diq010 = clean(merged['DIQ010'])                       # Y: not diabetes
df['Y'] = np.nan
df.loc[diq010 == 1, 'Y'] = 0
df.loc[diq010.isin([2, 3]), 'Y'] = 1

complete = df[['Z', 'X', 'Y']].dropna().astype(int)
pv = complete.value_counts().reset_index(name='count')
pv['prob'] = pv['count'] / pv['count'].sum()
diff = round(1.0 - pv['prob'].sum(), 10)
if abs(diff) > 0:
    pv.loc[pv['count'].idxmax(), 'prob'] += diff
pv = pv[['Z', 'X', 'Y', 'prob']].sort_values(['Z', 'X', 'Y']).reset_index(drop=True)
pv.to_csv(PV_PATH, index=False)

print(f'Complete cases: {len(complete)}')
print(pv.to_string(index=False))

Complete cases: 5833
 Z  X  Y     prob
 0  0  0 0.009258
 0  0  1 0.080747
 0  1  0 0.004115
 0  1  1 0.058803
 1  0  0 0.092062
 1  0  1 0.349906
 1  1  0 0.045088
 1  1  1 0.360021


## Potencies under each graph

The hull is unchanged but $Y$ gains a parent, so the potencies can move. We
recompute them from scratch under both graphs.


In [3]:
from autobound.DAG import DAG
from coupled_potency import NHANES_STRUCTURE, evaluate_potency, observational_bounds

ZY_STRUCTURE = 'Z->X, X->Y, Z->Y, U1->Z, U1->X, U2->X, U2->Y'
UNOB = 'U1,U2'

def make_dag(structure):
    d = DAG()
    d.from_structure(structure, unob=UNOB)
    return d

XY_CELLS = [(0, 0), (0, 1), (1, 0), (1, 1)]
A3_QUERIES = [f'X(Z={z})={x}&Y(Z={z})={y}' for z in (0, 1) for x, y in XY_CELLS]

CANDIDATES = [
    ('a3_tilde', A3_QUERIES),
    ('a4_tilde', ['Y(X=0)=1&Y(X=1)=0']),
    ('a5_tilde', ['Y(X=0)=1']),
    ('a6_tilde', ['Y(X=1)=1']),
    ('{a4_tilde, a6_tilde}', ['Y(X=0)=1&Y(X=1)=0', 'Y(X=1)=1']),
    ('{a4_tilde, a5_tilde}', ['Y(X=0)=1&Y(X=1)=0', 'Y(X=0)=1']),
]

def potencies(structure):
    dag = make_dag(structure)
    obs = observational_bounds(PV_PATH, dag=dag, workdir=str(ROOT))
    out = {'W_empty': obs['W_empty'], 'lb': obs['lb'], 'ub': obs['ub']}
    for label, queries in CANDIDATES:
        res = evaluate_potency(PV_PATH, queries, obs['W_empty'], dag=dag,
                               workdir=str(ROOT))
        assert res['run']['ok'], f'{label} did not solve'
        out[label] = res['potency']
    return out

t0 = time.time()
base = potencies(NHANES_STRUCTURE)
zy = potencies(ZY_STRUCTURE)
print(f'{time.time() - t0:.1f}s')

23.3s


In [4]:
rows = [{'quantity': 'W_empty', 'assumed': base['W_empty'], 'Z->Y': zy['W_empty']}]
rows += [{'quantity': label, 'assumed': base[label], 'Z->Y': zy[label]}
         for label, _ in CANDIDATES]
cmp_pot = pd.DataFrame(rows)
cmp_pot['diff'] = cmp_pot['Z->Y'] - cmp_pot['assumed']

print(f"theta assumed: [{base['lb']:.6f}, {base['ub']:.6f}]")
print(f"theta Z->Y   : [{zy['lb']:.6f}, {zy['ub']:.6f}]\n")
pd.set_option('display.float_format', lambda v: f'{v:.6f}')
print(cmp_pot.to_string(index=False))

# the ranking is what the selection depends on, so check it directly
COSTS = {'a3_tilde': 1, 'a4_tilde': 1, 'a5_tilde': 2, 'a6_tilde': 2,
         '{a4_tilde, a6_tilde}': 3, '{a4_tilde, a5_tilde}': 3}

def best(pot, B):
    aff = {k: pot[k] for k, c in COSTS.items() if c <= B}
    return max(aff, key=aff.get)

print()
for B in (1, 2, 3):
    b_assumed, b_zy = best(base, B), best(zy, B)
    flag = 'same' if b_assumed == b_zy else 'DIFFERENT'
    print(f'B={B}: assumed -> {b_assumed:<22} Z->Y -> {b_zy:<22} {flag}')


theta assumed: [-0.445064, 0.418745]
theta Z->Y   : [-0.479856, 0.520144]

            quantity  assumed     Z->Y     diff
             W_empty 0.863810 1.000001 0.136191
            a3_tilde 0.000000 0.000000 0.000000
            a4_tilde 0.439980 0.479856 0.039876
            a5_tilde 0.363291 0.468027 0.104736
            a6_tilde 0.500519 0.531973 0.031455
{a4_tilde, a6_tilde} 0.500519 0.531973 0.031455
{a4_tilde, a5_tilde} 0.776365 0.849477 0.073112

B=1: assumed -> a4_tilde               Z->Y -> a4_tilde               same
B=2: assumed -> a6_tilde               Z->Y -> a6_tilde               same
B=3: assumed -> {a4_tilde, a5_tilde}   Z->Y -> {a4_tilde, a5_tilde}   same


### Pruning decisions under $Z \to Y$

In [5]:
cg_base = CausalGraph(VARIABLES, EDGES, CONFOUNDERS)
cg_zy = CausalGraph(VARIABLES, EDGES + [('Z', 'Y')], CONFOUNDERS)

p_base, p_zy = pruning_table(cg_base), pruning_table(cg_zy)
print(f"R*_theta assumed : {sorted(cg_base.get_R_theta_star(QUERY_WORLDS))}")
print(f"R*_theta Z->Y    : {sorted(cg_zy.get_R_theta_star(QUERY_WORLDS))}\n")

print(f"{'label':<10}{'experiment':<16}{'assumed':>10}{'Z->Y':>8}")
for label, desc, _ in CANDIDATE_WORLDS:
    print(f'{label:<10}{desc:<16}{p_base[label]:>10}{p_zy[label]:>8}')
print(f'\nidentical: {p_base == p_zy}')

R*_theta assumed : ['U_X_Y', 'U_Z_X']
R*_theta Z->Y    : ['U_X_Y', 'U_Z_X']

label     experiment         assumed    Z->Y
a1_tilde  P(X | do(B))            ID      ID
a2_tilde  P(B | do(A))          Int.    Int.
a3_tilde  P(X,Y | do(z))          --      --
a4_tilde  P(y_x', y'_x)           --      --
a5_tilde  P(Y | do(x'))           --      --
a6_tilde  P(Y | do(x))            --      --

identical: True
